In [0]:
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col,
    to_timestamp,
    hour,
    dayofweek,
    month,
    current_timestamp,
    lit
)

source_table = "citibike_lakehouse.bronze.trips_raw"
target_table = "citibike_lakehouse.silver.trips"
process_timestamp = datetime.now(timezone.utc)

df_bronze = spark.table(source_table)

display(df_bronze.limit(10))
df_bronze.printSchema()

df_silver = (
    df_bronze
    .withColumn("started_at", to_timestamp(col("started_at"), "dd-MM-yyyy HH:mm:ss"))
    .withColumn("ended_at", to_timestamp(col("ended_at"), "dd-MM-yyyy HH:mm:ss"))
    .withColumn(
        "ride_duration_minutes", 
        (col("ended_at").cast("long") - col("started_at").cast("long")) / 60
    )
    .withColumn("hour", hour("started_at"))
    .withColumn("day_of_week", dayofweek("started_at"))
    .withColumn("month", month("started_at"))
    .withColumn("_processed_at", lit(process_timestamp))
    .withColumn("_inserted_at", current_timestamp())
)

df_silver = (
    df_silver
    .filter(col("started_at").isNotNull())
    .filter(col("ended_at").isNotNull())
    .filter(col("ride_duration_minutes") > 0)
    .dropDuplicates()
)

display(df_silver.limit(10))
df_silver.printSchema()


(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(target_table)
)